In [1]:
import sys

from absl import logging
from ferminet.utils import system
from ferminet import base_config
from ferminet import train
from ferminet.configs import atom

# Optional, for also printing training progress to STDOUT.
# If running a script, you can also just use the --alsologtostderr flag.
logging.get_absl_handler().python_handler.stream = sys.stdout
logging.set_verbosity(logging.INFO)


# Define H2 molecule
cfg = base_config.default()
cfg.system.electrons = (1,1)  # (alpha electrons, beta electrons)
cfg.system.molecule = [system.Atom('H', (0, 0, -1)), system.Atom('H', (0, 0, 1))]

# Set training parameters
cfg.batch_size = 4096
cfg.pretrain.iterations = 0
cfg.mcmc.burn_in = 0
cfg.optim.optimizer = 'minsr'


In [2]:
%load_ext autoreload
%autoreload 2

In [36]:
train.train(cfg, wandb_monitoring=False)

INFO:absl:Starting QMC with 1 XLA devices per host across 1 hosts.


cfg.optim.optimizer = 'minsr'


INFO:absl:No checkpoint found. Training new model.
INFO:absl:Burning in MCMC chain for 0 steps
INFO:absl:Completed burn-in MCMC steps
INFO:absl:Initial energy: -1.5480 E_h
2025-08-11 17:02:17.833643: W external/xla/xla/tsl/framework/bfc_allocator.cc:310] Allocator (GPU_0_bfc) ran out of memory trying to allocate 40.73GiB with freed_by_count=0. The caller indicates that this is not a failure, but this may mean that there could be performance gains if more memory were available.


XlaRuntimeError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 43736104960 bytes.

In [16]:
import jax
import jax.numpy as jnp

In [5]:
from ferminet import constants

In [6]:
batch_network = jax.vmap(
      logabs_network, in_axes=(None, 0, 0, 0, 0), out_axes=0
  )

In [80]:
sharded_key, subkeys = kfac_jax.utils.p_split(sharded_key)
#mcmc_keys, loss_keys = kfac_jax.utils.p_split(key)

NameError: name 'kfac_jax' is not defined

In [51]:
mcmc_step = constants.pmap(mcmc_step)
evaluate_loss = constants.pmap(evaluate_loss)
logabs_network = constants.pmap(logabs_network)

In [56]:
pmapped_batch_network = jax.pmap(batch_network, in_axes=(None, 0, 0, 0, 0))

In [17]:
data_positions = jnp.squeeze(data.positions, axis=0)
data_spins = jnp.squeeze(data.spins, axis=0)
data_atoms = jnp.squeeze(data.atoms, axis=0)
data_charges = jnp.squeeze(data.charges, axis=0)
params_ = jax.tree.map(lambda x: jnp.squeeze(x, axis=0), params)

In [ ]:
network_output = batch_network(params_, data_positions, data_spins, data_atoms, data_charges)

In [86]:
def f(params):
    # returns (batch_size,) outputs
    return batch_network(params, data_positions, data_spins, data_atoms, data_charges)

# Aggregate outputs to scalar loss
loss = jnp.mean(f(params_))  # or jnp.sum(...)

grads = jax.grad(lambda p: jnp.mean(f(p)))(params_)


In [89]:
param_sample = jax.tree_util.tree_leaves(grads)[0]
print(param_sample.shape)

(2, 32)


In [83]:
network_output.shape

(4096,)

In [11]:
val_grad = jax.value_and_grad(logabs_network, argnums=0)

In [12]:
batch_network_grad = jax.vmap(
      val_grad, in_axes=(None, 0, 0, 0, 0), out_axes=0
  )

In [26]:
from jax.flatten_util import ravel_pytree
import jax.numpy as jnp

minibatch_size = 256
blocks_ = []
for i in range(0, 4096, minibatch_size):
    # Compute gradients for the chunk
    grads_pytree = batch_network_grad(
        params_, 
        data_positions[i:i + minibatch_size], 
        data_spins[i:i + minibatch_size], 
        data_atoms[i:i + minibatch_size], 
        data_charges[i:i + minibatch_size]
    )
    
    # grads_pytree is a pytree with leaves shaped (batch_size=256, ...)
    
    # For each leaf, reshape from (batch_size, ...) to (batch_size, -1)
    flat_grads_leaves = [g.reshape((g.shape[0], -1)) for g in jax.tree_util.tree_leaves(grads_pytree)]
    
    # Concatenate all leaves along the last dimension -> (batch_size, n_param_chunk)
    flat_grads_chunk = jnp.concatenate(flat_grads_leaves, axis=1)
    blocks_.append(flat_grads_chunk @ flat_grads_chunk.T)
    
    #all_flat_grads.append(flat_grads_chunk)

# Concatenate all chunks along the batch dimension -> (4096, n_param)
#all_flat_grads = jnp.concatenate(all_flat_grads, axis=0)


2025-08-11 16:44:53.621762: W external/xla/xla/tsl/framework/bfc_allocator.cc:501] Allocator (GPU_0_bfc) ran out of memory trying to allocate 651.47MiB (rounded to 683115520)requested by op 
If the cause is memory fragmentation maybe the environment variable 'TF_GPU_ALLOCATOR=cuda_malloc_async' will improve the situation. 
Current allocation summary follows.
Current allocation summary follows.
2025-08-11 16:44:53.622109: W external/xla/xla/tsl/framework/bfc_allocator.cc:512] ****************************************_____***************__**********x******___*************x***_
E0811 16:44:53.622162   46424 pjrt_stream_executor_client.cc:2939] Execution of replica 0 failed: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 683115520 bytes. [tf-allocator-allocation-error='']


ValueError: RESOURCE_EXHAUSTED: Out of memory while trying to allocate 683115520 bytes.

In [31]:
blocks.shape

AttributeError: 'list' object has no attribute 'shape'

In [30]:
blocks[0].shape

(128, 128)

In [ ]:
128 * 128 * 32

In [33]:
4096 / 32

128.0

In [32]:
import jax.numpy as jnp

# Suppose you have row_blocks_list where each element is (32, b, b)
# and there are 32 such elements, one for each block-row.
block_matrix = jnp.stack(blocks)   # shape (32, 32, b, b)

# Rearrange into (32*b, 32*b)
grunter = (block_matrix
           .transpose(0, 2, 1, 3)
           .reshape(32*b, 32*b))


ValueError: axis 3 is out of bounds for array of dimension 3

In [29]:
grunter.shape

(4096, 128)

### network_output = pmapped_batch_network(params, data.positions, data.spins, data.atoms, data.charges)

In [ ]:
network_output.shape

In [64]:
import jax.numpy as jnp

In [65]:
data_positions = jnp.squeeze(data.positions, axis=0)

In [66]:
data_positions.shape

(4096, 6)

In [41]:
loss, aux_loss = evaluate_loss(params, sharded_key, data)

In [45]:
loss

Array([-1.2800125], dtype=float32)

In [46]:
aux_loss.local_energy.shape

(1, 4096)